# Stage 1 — Ingest & Normalize (Vision)

**Deliverable of this notebook:** `data/raw/` — one structured,
content-verified text file per offer.

| Stage | Notebook | Output |
|---|---|---|
| 1 — Ingest & Normalize (this one) | `01-ingest-normalize.ipynb` | `data/raw/{AG}.txt` (+ `.ref.txt`, `.json`) |
| 2 — Sanitize & Extract | `02-sanitize-extract.ipynb` | `data/redacted/text/`, `data/extracted/` |
| 3 — Build Index | `03-build-index.ipynb` | `data/db/chroma/`, `data/db/sql/` |

**Flow:**
1. **Ingestion (intermediate step):** pdfplumber `layout=False` → flat
   reference text per offer. It serves only as the deterministic reference
   for the fidelity check — the actual content extraction is done by the
   vision model.
2. **Vision normalization (variant 1c, selected in `vision-test.ipynb`):**
   The page images (@150 DPI) go directly to the vision model
   (`qwen3.8:27b`, thinking OFF, temperature 0). The model transcribes the
   document and structures the layout (label/value lines, table rows,
   address blocks, page markers) — without changing the content.
3. **Fidelity check + repair loop:** deterministic word-multiset diff
   (vision output vs. pdfplumber reference) + page-marker completeness.
   Violations are sent back to the model with the exact token list
   (max. 2 repair attempts). **STRICT:** offers with remaining violations
   are not written and abort the run at the end.

**Resumable:** Finished offers are cached (`{AG}.json` + `{AG}.txt`) and
skipped on re-runs. For a fresh start: set `CLEAN_SLATE = True` in the
clean-slate cell.

**Privacy:** `source/offers/` is only read. PII masking does not happen
here — that is Stage 2.


## Setup — Environment, Paths & LLM Client

Load `.env`, define paths, create the OpenAI client (thinking OFF, temp 0).

In [ ]:
import os, re, json, time, base64, shutil
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load env files (walk up: notebooks/ -> repo root). .env.example provides
# the defaults, .env overrides them (same layering as the app's config.py).
for path in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    example, env = path / ".env.example", path / ".env"
    if example.exists():
        load_dotenv(example, override=False)
        print(f"✅ Loaded defaults from {example}")
    if env.exists():
        load_dotenv(env, override=True)
        print(f"✅ Loaded .env from {env}")
    if example.exists() or env.exists():
        break

# Secrets layer (outside the repo, chmod 600): highest priority, keeps API
# keys out of the workspace (same as the app's config.py).
_secrets = Path.home() / ".config" / "rag-quote-history" / "secrets.env"
if _secrets.exists():
    load_dotenv(_secrets, override=True)
    print(f"\u2705 Loaded secrets from {_secrets}")

LLM_BASE_URL = os.getenv("LLM_BASE_URL", "")
LLM_MODEL = os.getenv("LLM_MODEL", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "")

def redact_url(url: str) -> str:
    """Mask IP address and port in a URL for display (e.g. http://*.*.*.*:****/v1)."""
    url = re.sub(r"\d{1,3}(?:\.\d{1,3}){3}", "*.*.*.*", url)
    return re.sub(r":\d+", ":****", url)

DEMO_DIR = Path.cwd().parent
SOURCE_DIR = DEMO_DIR / "source" / "offers"
DATA_DIR = DEMO_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

n_pdfs = len([f for f in SOURCE_DIR.iterdir() if f.suffix.lower() == ".pdf"])
print(f"📁 Source: {SOURCE_DIR} ({n_pdfs} PDFs)")
print(f"📁 Output: {RAW_DIR}")
print(f"🤖 LLM: {LLM_MODEL} at {redact_url(LLM_BASE_URL)}")

In [ ]:
# Clean slate for THIS stage only: data/raw/ is emptied.
# data/redacted/, data/extracted/, data/db/ belong to later stages — untouched.
# Set CLEAN_SLATE = False when resuming a partially completed run
# (the loop below skips finished offers via the cache).
CLEAN_SLATE = False

if CLEAN_SLATE and RAW_DIR.exists():
    shutil.rmtree(RAW_DIR)
RAW_DIR.mkdir(parents=True, exist_ok=True)
print(f"🧹 Clean slate: {RAW_DIR} {'emptied' if CLEAN_SLATE else 'kept (resume mode)'}")


In [ ]:
raw_client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)
print(f"✅ LLM client ready: {LLM_MODEL} (thinking OFF, temperature 0)")


## Step 1: Ingestion (PDF → reference text)

**pdfplumber `layout=False`** — flat text per page. This text is an
*intermediate step*: it serves as the deterministic reference for the
fidelity check in Step 2 (and for the alpha-ratio gate). The actual content
extraction is done by the vision model — no layout parsing happens here.

Per PDF:
- **AG####** is extracted from the filename (primary key)
- **Non-offer files** are excluded and logged
  (`KVA_`, `LEISTUNGSUEBERSICHT_`, `ZUSAMMENFASSUNG`)
- **Duplicates** (same AG####, different file): first wins, the rest are logged
- **Alpha-ratio gate** (0.30): pure scan/image PDFs are skipped and logged


In [ ]:
import pdfplumber

NON_OFFER_PATTERNS = ["KVA_", "LEISTUNGSUEBERSICHT_", "ZUSAMMENFASSUNG"]
AG_RE = re.compile(r"AG\d{4}")
ALPHA_RATIO_MIN = 0.30

def extract_flat_text(page) -> str:
    """Flat text extraction (layout=False) — reference for the fidelity check."""
    raw = page.extract_text(layout=False) or ""
    lines, prev_blank = [], False
    for ln in raw.splitlines():
        ln = ln.strip()
        blank = (ln == "")
        if blank and prev_blank:
            continue
        lines.append(ln)
        prev_blank = blank
    while lines and not lines[0]:
        lines.pop(0)
    while lines and not lines[-1]:
        lines.pop()
    return "\n".join(lines)

def _fix_glued_number(text):
    """Repair a pdfplumber artifact where a space lands inside a number:
    'von1 4' -> 'von 14'. The single digit glued to the word is the first
    digit of the number; the rest follows after the stray space. Only acts
    on letter + single-digit + space + digits (the artifact shape), so
    legitimate text (e.g. 'von 14', 'AG0086 4') is untouched."""
    return re.sub(r"([A-Za-z\u00c4\u00d6\u00dc\u00e4\u00f6\u00fc\u00df])(\d) (\d+)",
                  r"\1 \2\3", text)

offers_ref = {}      # AG#### -> pdfplumber reference text
offer_to_pdf = {}    # AG#### -> pdf path
excluded, gate_skips, read_errors, duplicates = [], [], [], []

pdf_files = sorted(f for f in SOURCE_DIR.iterdir() if f.suffix.lower() == ".pdf")
for pdf_path in pdf_files:
    m = AG_RE.search(pdf_path.stem)
    if not m:
        excluded.append((pdf_path.name, "no AG#### in filename"))
        continue
    ag = m.group(0)
    if any(p in pdf_path.stem.upper() for p in NON_OFFER_PATTERNS):
        excluded.append((pdf_path.name, "non-offer file"))
        continue
    if ag in offer_to_pdf:
        duplicates.append((pdf_path.name, offer_to_pdf[ag].name))
        continue
    try:
        with pdfplumber.open(str(pdf_path)) as pdf:
            text = "\n\n".join(extract_flat_text(p) for p in pdf.pages)
        text = _fix_glued_number(text)
    except Exception as e:
        read_errors.append((pdf_path.name, str(e)))
        continue
    alpha = sum(c.isalpha() for c in text) / max(len(text), 1)
    if alpha < ALPHA_RATIO_MIN:
        gate_skips.append((pdf_path.name, f"alpha ratio {alpha:.2f} < {ALPHA_RATIO_MIN}"))
        continue
    offers_ref[ag] = text
    offer_to_pdf[ag] = pdf_path
    (RAW_DIR / f"{ag}.ref.txt").write_text(text)

print(f"✅ {len(offers_ref)} offers ingested")
if excluded:
    print(f"⏭️  excluded ({len(excluded)}):")
    for n, r in excluded: print(f"   {n} — {r}")
if duplicates:
    print(f"⏭️  duplicates, first wins ({len(duplicates)}):")
    for n, k in duplicates: print(f"   {n} (kept {k})")
if gate_skips:
    print(f"⚠️  gate skips ({len(gate_skips)}):")
    for n, r in gate_skips: print(f"   {n} — {r}")
if read_errors:
    print(f"❌ read errors ({len(read_errors)}):")
    for n, r in read_errors: print(f"   {n} — {r}")
assert offers_ref, "No offers ingested — check source directory"


## Step 2: Vision Normalization (page images → structured text)

The LLM sees each page as an image and transcribes + normalizes layout:
header blocks, table rows, page markers. The pdfplumber flat text from
Step 1 serves as a fidelity reference (no LLM in the ground truth).

- **Header read order (semantic blocks):** the header is treated as a set
  of semantic blocks, not columns — output order: (a) sender name/title,
  (b) sender address, (c) recipient (customer) block, (d) letterhead
  details (office/contact, court/tax office, bank). Only blocks that
  actually exist in the document are output — missing blocks are skipped,
  never invented. This works for both two-column header variants (sender
  left or sender right) and prevents the model from interleaving lines of
  side-by-side blocks.

In [ ]:
import pymupdf as fitz
from collections import Counter

VISION_NORMALIZE_PROMPT = """This is a German post-production project offer (all pages as images).

Transcribe the document and normalize ONLY the layout, NO content:
1. Label/value fields (e.g. Angebotsnr., Kundennr., Datum, gültig bis) into
   one line per field: "Datum: 16.09.2020"
2. Table rows (Pos., Bezeichnung, Beschreibung, Menge, Einheit, Preise)
   into ONE line per position.
3. Address/info blocks (customer, sender, bank, tax office) as separate
   blocks, each in its own sequence of lines.
   HEADER READ ORDER: the header is a set of semantic blocks, not
   columns. Identify the blocks that ACTUALLY EXIST in the document and
   output them in this relative order:
   (a) SENDER: the company/person name and title line(s) at the top
       of the header.
   (b) SENDER ADDRESS: the sender's own address line(s).
   (c) RECIPIENT: the customer name and address block.
   (d) LETTERHEAD DETAILS: office/contact lines (OFFICE/FON/MAIL/WEB),
       court/tax office, bank details (IBAN/BIC).
   Output each document line exactly ONCE. If the sender name and address
   share a single line in the document (e.g. "Name | Street | City"),
   keep them on that one line — do NOT also emit the name as a separate
   line.
   Only output blocks that are present in the document — if a block is
   missing, skip it. NEVER invent, complete or guess block content:
   every line in the output must be a line that exists in the document.
   Each block is one contiguous sequence of lines, separated from the
   next block by an empty line. Never interleave lines of two blocks,
   and never move a line to a different block than in the original.
4. Empty lines between logical blocks.
5. PAGE MARKERS: Start each page with its own line
   "[Seite 1 von N]", "[Seite 2 von N]", ... — even if the page content
   (e.g. footer) repeats. Repeated footers belong to their respective page
   and are NOT merged.

HARD RULES:
- Do not add, change or drop any word, number, amount or date.
- No translations, no corrections, no additions.
- Output = the document content, only re-shaped. No commentary, no markdown fences."""

REPAIR_PROMPT = """Your previous output violated the HARD RULES:
words/numbers were changed or dropped compared to the original.

Changed tokens (original → your output):
{changes}

Here is your previous output:
---
{prev}
---

Fix ONLY the changed tokens back to the original.
Change nothing else. Output ONLY the corrected full text."""

STRICT = True      # True: failed offers are not written, run aborts at the end
MAX_ATTEMPTS = 2   # LLM repair attempts after the initial generation

def render_page_images(pdf_file, dpi=150):
    doc = fitz.open(str(pdf_file))
    imgs = []
    for page in doc:
        pix = page.get_pixmap(dpi=dpi)
        imgs.append({"type": "image_url",
                     "image_url": {"url": f"data:image/png;base64,{base64.b64encode(pix.tobytes('png')).decode()}"}})
    n_pages = len(doc)
    doc.close()
    return imgs, n_pages

def _chat(messages, temperature=0.0, max_tokens=4000):
    resp = raw_client.chat.completions.create(
        model=LLM_MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    return resp.choices[0].message.content.strip(), resp.usage

def _word_tokens(text):
    """Word tokens (containing at least one alnum char), page markers
    stripped. Line-break hyphenation is reconciled in content_violations
    (not here) because the model may re-flow the fragments, so they are
    not always adjacent."""
    return [t for t in re.sub(r"\[Seite \d+ von \d+\]", "", text).split()
            if any(ch.isalnum() for ch in t)]


def _is_left_frag(t):
    """A line-break left fragment: ends in '-' with a letter or digit before
    it, and at least one letter in the fragment (so pure numbers like '4-'
    are not treated as fragments, but 'P3-' is)."""
    return (t.endswith("-") and len(t) > 1 and t[-2].isalnum()
            and any(c.isalpha() for c in t[:-1]))


def _join_variants(xf, y):
    """Possible joins of a left fragment 'X-' with a following token 'Y'.
    The model may keep the hyphen ('X-Y'), drop it ('XY'), or drop it and
    lowercase the first letter of Y ('Xy') when it reads the split word as
    one word. All are the same content, so all are accepted."""
    variants = {xf + y, xf[:-1] + y}
    if y[:1].isupper():
        variants.add(xf[:-1] + y[0].lower() + y[1:])
    return variants


def _num_norm(t):
    """Normalize a numeric token's decimal separator (',' <-> '.') for
    comparison — the model may render a German decimal comma as a period
    ('29,97' -> '29.97'). Returns the normalized form, or None if the token
    is not a plain number (digits with at most one separator)."""
    s = t.strip(".,;:")
    if not s or not all(c.isdigit() or c in "., " for c in s):
        return None
    s = s.replace(" ", "")
    if s.count(",") + s.count(".") > 1:
        return None
    return s.replace(",", ".")


def content_violations(raw, out):
    """Dup-tolerant multiset diff on WORD tokens, hyphenation-aware.

    - Page markers ([Seite X von Y]) are REQUIRED by rule 5, so they are
      stripped from the output before the diff — they are not violations.
    - Pure punctuation tokens (|, -, ·, *, ...) are ignored: separator
      style is layout, not content. Words, numbers, amounts and dates
      are checked strictly.
    - Line-break hyphenation is layout, not content: a word split across a
      line break ("Refe-" + "renzmonitoring.") equals the joined word
      ("Referenzmonitoring."), wherever the model re-flowed the fragments,
      and whether the model kept or dropped the hyphen / capital.
      Reconciled position-independently (both directions) before the diff.
    - dropped: word tokens missing from the output entirely (repeated
      footers must not false-alarm — only flagged if count_out == 0)
    - added:   word tokens in the output that are not in the input
      (with multiplicity)
    """
    c_raw, c_out = Counter(_word_tokens(raw)), Counter(_word_tokens(out))
    dropped = Counter(c_raw - c_out)
    added = Counter(c_out - c_raw)
    # Reconcile line-break hyphen fragments position-independently: a left
    # fragment "X-" + token "Y" on one side == the joined word on the other.
    changed = True
    while changed:
        changed = False
        for xf in [t for t in dropped if dropped[t] > 0 and _is_left_frag(t)]:
            for y in [t for t in dropped if t != xf and dropped[t] > 0
                      and t[:1].isalpha()]:
                for joined in _join_variants(xf, y):
                    if added.get(joined, 0) > 0:
                        dropped[xf] -= 1
                        dropped[y] -= 1
                        added[joined] -= 1
                        changed = True
                        break
                if changed:
                    break
        for xf in [t for t in added if added[t] > 0 and _is_left_frag(t)]:
            for y in [t for t in added if t != xf and added[t] > 0
                      and t[:1].isalpha()]:
                for joined in _join_variants(xf, y):
                    if dropped.get(joined, 0) > 0:
                        added[xf] -= 1
                        added[y] -= 1
                        dropped[joined] -= 1
                        changed = True
                        break
                if changed:
                    break
        # Reconcile numeric tokens that differ only in the decimal separator
        # (',' vs '.'): layout/format, not content.
        for d in [t for t in dropped if dropped[t] > 0]:
            dn = _num_norm(d)
            if dn is None:
                continue
            for a in [t for t in added if added[t] > 0]:
                if _num_norm(a) == dn:
                    dropped[d] -= 1
                    added[a] -= 1
                    changed = True
                    break
            if changed:
                break
    dropped = sorted(t for t in dropped if dropped[t] > 0 and c_out.get(t, 0) == 0)
    added = sorted(list(added.elements()))
    return dropped, added


def _lev(a, b):
    """Levenshtein distance, case-insensitive."""
    a, b = a.lower(), b.lower()
    if a == b:
        return 0
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

def _join_kept_line_breaks(out, drop_pool, add_pool):
    """Join line breaks the model KEPT (reverse of _dehyphenate).

    The dehyphenated reference contains the joined word
    ("Referenzmonitoring.") while the output still has the split pair
    on consecutive lines ("Refe-" + "renzmonitoring."); the pair is
    replaced by the joined reference form (lowercase follows: no
    hyphen; uppercase follows: hyphen kept). Only acts when the joined
    form is missing from the output (in drop_pool) and both split
    tokens are extra (in add_pool).
    Returns (text, joins) — joins: (token1, token2, joined) tuples.
    """
    lines = out.splitlines()
    joins = []
    i = 0
    while i < len(lines) - 1:
        ln, nxt = lines[i], lines[i + 1]
        if (ln.endswith("-") and len(ln) > 1 and ln[-2].isalpha()
                and nxt[:1].isalpha()):
            joined = ln + nxt if nxt[0].isupper() else ln[:-1] + nxt
            t1, t2 = ln.rsplit(" ", 1)[-1], nxt.split(" ", 1)[0]
            if (drop_pool.get(joined, 0) > 0
                    and add_pool.get(t1, 0) > 0 and add_pool.get(t2, 0) > 0):
                pre = ln[:len(ln) - len(t1)].rstrip()
                post = nxt[len(t2):].lstrip()
                lines[i] = (pre + " " if pre else "") + joined
                if post:
                    lines[i + 1] = post
                else:
                    del lines[i + 1]
                drop_pool[joined] -= 1
                add_pool[t1] -= 1
                add_pool[t2] -= 1
                joins.append((t1, t2, joined))
                continue
        i += 1
    return "\n".join(lines), joins

def _remove_duplicate_prefix_lines(out, add_pool):
    """Remove a line that is a pure duplicate of part of another line.

    The model sometimes splits a name+address line into two lines,
    duplicating the name: line A = "Nordlicht Digital Imaging" (name
    alone) and line B = "Nordlicht Digital Imaging | Hafenweg 12
    | 24105 Nordhafen" (the full original line). Line A's word tokens are a
    STRICT subset of line B's (line B has at least one token line A does
    not) and are all 'extra' (present in add_pool), so line A is a pure
    duplicate and is removed. Two guards keep it safe:
    - only a line whose tokens are ALL extra is ever removed, so a
      legitimate line (whose tokens match the reference) is never touched;
    - the subset must be STRICT: identical repeated lines (e.g. a footer
      repeated on several pages) are never removed, and a pure 1:1
      substitution diff (equal counts) is left to
      final_deterministic_replace.
    Returns (text, removed_lines)."""
    lines = out.splitlines()
    removed = []
    while True:
        line_tokens = [Counter(_word_tokens(ln)) for ln in lines]
        target = None
        for i, lt in enumerate(line_tokens):
            if not lt:
                continue
            # every token of line i must be extra (in add_pool, enough count)
            if any(add_pool.get(t, 0) < c for t, c in lt.items()):
                continue
            # line i's tokens must be a STRICT subset of some OTHER
            # line's tokens (that line has at least one token line i
            # does not) — identical repeated lines are never removed
            if not any(j != i and (line_tokens[j] - lt)
                       and all(line_tokens[j].get(t, 0) >= c
                               for t, c in lt.items())
                       for j in range(len(lines))):
                continue
            target = i
            break
        if target is None:
            break
        lt = line_tokens[target]
        for t, c in lt.items():
            add_pool[t] -= c
        removed.append(lines[target])
        del lines[target]
    return "\n".join(lines), removed

def deterministic_repair(raw, out):
    """Restore original tokens in the model output — pure Python, no LLM.

    Pairs dropped (original) tokens with added (output) tokens:
    0. Duplicate prefix-line removal: the model splits a name+address
       line into two lines, duplicating the name; the name-only line is
       removed (its tokens are a subset of the full line's and all extra).
    1. Reverse hyphen merges: the model KEPT a line break (output
       "Refe-\nrenzmonitoring.") while the dehyphenated reference has
       the joined word; the pair is joined back into the reference form.
    2. Hyphen merges: the model joins line-broken hyphenated words,
       keeping or dropping the line-break hyphen (original "Digital-" +
       "Werbebannern," -> output "Digital-Werbebannern," or
       "DigitalWerbebannern,"; original "Refe-" + "renzmonitoring." ->
       output "Referenzmonitoring."); the output token is replaced by
       the correctly joined form (lowercase follows: no hyphen; uppercase
       follows: hyphen kept). The following token is matched against the
       WHOLE original (it may be a common word that is not itself
       dropped, e.g. "Kampagne.").
    3. Typos: the model "corrects" typos of the original (original
       "Inahalte" -> output "Inhalte"); the ORIGINAL token is restored.
       Edit distance <= 2 (case-insensitive); the token must contain at
       least one letter — pure numbers/amounts are NEVER fuzzy-paired.
    The pairing pool is the FULL multiset difference (c_raw - c_out), so
    partially converted tokens (e.g. original "u.A." x2, output "u.A." x1
    + "u.a." x1) are paired as well.
    Returns (repaired_text, replacements) where replacements is a list of
    (output_token, restored_original, kind) tuples, in order of occurrence.
    """
    c_raw, c_out = Counter(_word_tokens(raw)), Counter(_word_tokens(out))
    drop_pool = Counter(c_raw - c_out)
    add_pool = Counter(c_out - c_raw)
    replacements = []
    # 0) duplicate prefix-line removal: the model splits a name+address
    # line into two lines, duplicating the name; the name-only line's
    # tokens are a subset of the full line's and are all extra, so it is
    # removed. Runs even when nothing is dropped (drop_pool empty) — the
    # duplication only ever ADDS tokens.
    out, removed = _remove_duplicate_prefix_lines(out, add_pool)
    if removed:
        for ln in removed:
            replacements.append((ln, "", "dup-line"))
        c_raw, c_out = Counter(_word_tokens(raw)), Counter(_word_tokens(out))
        drop_pool = Counter(c_raw - c_out)
        add_pool = Counter(c_out - c_raw)
    if not drop_pool or not add_pool:
        return out, replacements
    # 1) reverse hyphen merges: the model KEPT the line break (output
    # "Refe-\nrenzmonitoring.") while the dehyphenated reference has the
    # joined word ("Referenzmonitoring."); join the pair back.
    out, joins = _join_kept_line_breaks(out, drop_pool, add_pool)
    if joins:
        for t1, t2, joined in joins:
            replacements.append((f"{t1}+{t2}", joined, "hyphen-join"))
        c_raw, c_out = Counter(_word_tokens(raw)), Counter(_word_tokens(out))
        drop_pool = Counter(c_raw - c_out)
        add_pool = Counter(c_out - c_raw)
        if not drop_pool or not add_pool:
            return out, replacements
    fixes = {}  # output token -> list of (original replacement, kind) per occurrence
    # 2) hyphen merges: output token = dropped hyphen token + following original token
    for a in sorted(add_pool, key=len, reverse=True):
        for _ in range(add_pool[a]):
            candidates = [d for d in drop_pool
                          if d.endswith("-") and len(d) > 1 and drop_pool[d] > 0
                          and a.startswith(d.rstrip("-"))]
            if not candidates:
                continue
            d1 = max(candidates, key=len)          # longest hyphen prefix wins
            rest = a[len(d1.rstrip("-")):]
            rest_orig = None
            if rest in c_raw:
                rest_orig = rest
            else:
                best, best_d = None, 3
                for t in c_raw:
                    if any(c.isalpha() for c in t):
                        d = _lev(rest, t)
                        if d < best_d:
                            best, best_d = t, d
                if best is not None:
                    rest_orig = best
            if rest_orig is None:
                continue
            joiner = "-" if rest_orig[:1].isupper() else ""
            fixes.setdefault(a, []).append(
                (d1.rstrip("-") + joiner + rest_orig, "hyphen-merge"))
            drop_pool[d1] -= 1
    # 3) typo pairs (edit distance <= 2, letters required)
    for a in sorted(add_pool):
        if a in fixes or not any(c.isalpha() for c in a):
            continue
        for _ in range(add_pool[a]):
            best, best_d = None, 3
            for d in [d for d in drop_pool if drop_pool[d] > 0]:
                dist = _lev(a, d)
                if dist < best_d:
                    best, best_d = d, dist
            if best is None:
                break
            fixes.setdefault(a, []).append((best, "typo"))
            drop_pool[best] -= 1
    if not fixes:
        return out, replacements
    def _repl(m):
        t = m.group(0)
        if t in fixes and fixes[t]:
            new, kind = fixes[t].pop(0)
            replacements.append((t, new, kind))
            return new
        return t
    return re.sub(r"\S+", _repl, out), replacements

def final_deterministic_replace(raw, out):
    """Deterministic substitution fix — pure Python, no LLM.

    Only acts when the remaining diff is a PURE 1:1 token substitution
    (equal total counts on both sides — no token missing or extra).
    Example: the model "corrects" a non-standard original word into a real
    one ("Werbeausspielungen" -> "Werbeeinspielungen", edit distance 3,
    beyond the typo threshold). The added tokens are replaced by the
    dropped originals via exact string replacement; pairs are matched
    greedily by edit distance. The multiset diff guarantees the result
    matches the reference exactly; if no perfect pairing exists, the text
    is returned unchanged (the strict check still fails).
    Returns (text, replacements) — replacements: (added, dropped) tuples.
    """
    c_raw, c_out = Counter(_word_tokens(raw)), Counter(_word_tokens(out))
    drop_pool = Counter(c_raw - c_out)
    add_pool = Counter(c_out - c_raw)
    if not drop_pool or not add_pool:
        return out, []
    if sum(drop_pool.values()) != sum(add_pool.values()):
        return out, []  # missing or extra tokens — only the LLM can fix it
    pairs = sorted((_lev(a, d), a, d) for a in add_pool for d in drop_pool)
    fixes, used_add, used_drop = {}, Counter(), Counter()
    for _dist, a, d in pairs:
        if used_add[a] >= add_pool[a] or used_drop[d] >= drop_pool[d]:
            continue
        fixes.setdefault(a, []).append(d)
        used_add[a] += 1
        used_drop[d] += 1
    if used_add != add_pool or used_drop != drop_pool:
        return out, []  # no perfect pairing — leave it to the strict check
    replacements = []
    def _repl(m):
        t = m.group(0)
        if t in fixes and fixes[t]:
            new = fixes[t].pop(0)
            replacements.append((t, new))
            return new
        return t
    return re.sub(r"\S+", _repl, out), replacements

def show_flags(attempt, dropped, added, pages_ok):
    print(f"--- 🚩 FLAGGED (attempt {attempt}) " + "-" * 40)
    if dropped:
        print(f"   dropped ({len(dropped)}): {dropped}")
    if added:
        print(f"   added   ({len(added)}): {added}")
    if not dropped and not added:
        print("   no content violations")
    print(f"   page markers: {'✅ complete' if pages_ok else '🚩 missing'}")
    print("-" * 70)

def check_output(raw, out, n_pages, attempt=0):
    """Deterministic guards: content diff + page-marker completeness."""
    dropped, added = content_violations(raw, out)
    found = sorted(set(int(m) for m in re.findall(r"\[Seite (\d+) von \d+\]", out)))
    pages_ok = found == list(range(1, n_pages + 1))
    show_flags(attempt, dropped, added, pages_ok)
    return dropped, added, pages_ok

def vision_normalize_checked(model, images, n_pages, raw, strict=STRICT, max_attempts=MAX_ATTEMPTS):
    """Generate → deterministic repair → check → deterministic
    substitution → LLM repair (only for missing/extra tokens). Thinking OFF.

    The LLM repair is ONLY spent when tokens are missing or extra
    (unequal counts) — only the model can re-read the page and place a
    missing token. Pure 1:1 substitutions (equal counts) are restored
    deterministically for free, BEFORE any LLM budget is spent, and again
    after the budget is exhausted as a safety net.

    Returns (text, clean, n_llm_repairs) — n_llm_repairs counts how many
    times the LLM repair loop had to run (0 = clean without LLM repair).
    """
    prompt = VISION_NORMALIZE_PROMPT.replace("von N]", f"von {n_pages}]")
    out, usage = _chat([{"role": "user", "content": [{"type": "text", "text": prompt}, *images]}])
    print(f"   attempt 0: {len(out)} chars, tokens in/out: "
          f"{(usage.prompt_tokens, usage.completion_tokens) if usage else '?'}")
    n_llm_repairs = 0
    for attempt in range(1, max_attempts + 2):
        out, det_fixes = deterministic_repair(raw, out)
        if det_fixes:
            print(f"   🔧 deterministic repair: {len(det_fixes)} token(s) restored from original")
            for old, new, kind in det_fixes:
                print(f"      [{kind}] {old!r} → {new!r}")
        dropped, added, pages_ok = check_output(raw, out, n_pages, attempt=attempt - 1)
        if not dropped and not added and pages_ok:
            return out, True, n_llm_repairs
        # Free deterministic substitution (no LLM budget): only acts when
        # the remaining diff is a pure 1:1 token substitution — no token
        # is missing or extra. The LLM is deliberately not involved: for
        # such pairs it would just "correct" the original back.
        if dropped or added:
            out, final_fixes = final_deterministic_replace(raw, out)
            if final_fixes:
                print(f"   🔧 deterministic substitution: {len(final_fixes)} token(s) restored from original")
                for old, new in final_fixes:
                    print(f"      [final-replace] {old!r} → {new!r}")
                dropped, added, pages_ok = check_output(raw, out, n_pages, attempt=attempt - 1)
                if not dropped and not added and pages_ok:
                    return out, True, n_llm_repairs
        if attempt > max_attempts or not (dropped or added):
            break  # LLM repair budget exhausted, or only page markers missing
        changes = "\n".join(f"  - Original: {t!r}  →  your output: {t!r} (restore!)" for t in dropped)
        if added:
            changes += "\n" + "\n".join(f"  - Added (remove): {t!r}" for t in added)
        print(f"   → LLM repair {attempt - 1}: {len(dropped)} dropped, {len(added)} added")
        n_llm_repairs += 1
        out, usage = _chat([
            {"role": "user", "content": REPAIR_PROMPT.format(changes=changes, prev=out)},
        ])
        print(f"   attempt {attempt}: {len(out)} chars")
    # Safety net: the last LLM output may have left a pure 1:1
    # substitution behind — restore it deterministically (free).
    if dropped or added:
        out, final_fixes = final_deterministic_replace(raw, out)
        if final_fixes:
            print(f"   🔧 deterministic substitution: {len(final_fixes)} token(s) restored from original")
            for old, new in final_fixes:
                print(f"      [final-replace] {old!r} → {new!r}")
            dropped, added, pages_ok = check_output(raw, out, n_pages, attempt=max_attempts + 1)
    clean = not dropped and not added and pages_ok
    if strict and not clean:
        raise AssertionError(
            f"Content-fidelity check FAILED after {max_attempts} repair attempts "
            f"(strict mode). dropped: {dropped} | added: {added} | pages_ok: {pages_ok}"
        )
    return out, clean, n_llm_repairs

In [ ]:
# Debug switch: LIMIT = 3 runs the loop for the first 3 offers only
# (alphabetical order). Set LIMIT = None for the full run.
LIMIT = None

from datetime import datetime

DATE_RE = r"\d{2}\.\d{2}\.\d{4}"

def extract_datum(ref):
    """Offer date from the pdfplumber reference (ground truth, no LLM).

    Three layouts exist in the real data:
    1. inline (2026 offers):  "Peer-Span GmbH Angebotsdatum: 20.05.2026"
    2. stacked: label line "Angebotsnr.: Kundennr.: Datum: gültig bis:"
       + value line "AG0002 10002 23.06.2020 23.07.2020" (value at the
       label's column index)
    3. stacked + title: a "Kostenvoranschlag" line between label and value
    Multi-token values (e.g. USt-IdNr. "DE32 494 7171") shift columns to
    the right, so the first date token at/after the expected position wins.
    "gültig bis" is a separate label and never matched.
    """
    m = re.search(rf"Angebotsdatum:\s*({DATE_RE})", ref)
    if m:
        return m.group(1)
    lines = ref.splitlines()
    for i, ln in enumerate(lines):
        labels = re.findall(r"([\w.]+):", ln)
        if "Datum" not in labels:
            continue
        idx = labels.index("Datum")
        for j in (i + 1, i + 2):  # value line 1 or 2 lines below
            if j >= len(lines):
                break
            for v in lines[j].split()[idx:]:
                if re.fullmatch(DATE_RE, v):
                    return v
    return None

def to_iso(d):
    """'28.01.2022' -> '2022-01-28' (ISO 8601); None/invalid -> None."""
    if not d:
        return None
    try:
        return datetime.strptime(d, "%d.%m.%Y").date().isoformat()
    except ValueError:
        return None

failed = []
todo = sorted(offers_ref)
if LIMIT is not None:
    todo = todo[:LIMIT]
    print(f"🔬 DEBUG MODE: running first {len(todo)} offers only: {todo}")
for ag in todo:
    status_path = RAW_DIR / f"{ag}.json"
    out_path = RAW_DIR / f"{ag}.txt"
    if status_path.exists() and out_path.exists():
        st = json.loads(status_path.read_text())
        if st.get("clean"):
            print(f"♻️  {ag}: cached ({st['chars']} chars)")
            continue
    ref = offers_ref[ag]
    images, n_pages = render_page_images(offer_to_pdf[ag])
    try:
        out, clean, n_llm_repairs = vision_normalize_checked(LLM_MODEL, images, n_pages, ref)
    except AssertionError as e:
        failed.append((ag, str(e)))
        continue
    out_path.write_text(out)
    datum = to_iso(extract_datum(ref))
    status_path.write_text(json.dumps({
        "angebot_id": ag,
        "clean": clean,
        "datum": datum,
        "n_pages": n_pages,
        "chars": len(out),
        "ref_chars": len(ref),
        "llm_repairs": n_llm_repairs,
        "pdf": offer_to_pdf[ag].name,
    }, ensure_ascii=False, indent=2))
    print(f"✅ {ag}: {len(ref)} → {len(out)} chars, clean: {clean}, "
          f"datum: {datum or '❓ not found'}, llm_repairs: {n_llm_repairs}")

print("=" * 70)
print(f"📊 {len(todo) - len(failed)}/{len(todo)} offers normalized"
      + (f" (debug limit: {LIMIT})" if LIMIT is not None else ""))
if failed:
    print("❌ FAILED (not written):")
    for ag, err in failed:
        print(f"   {ag}: {err}")
    raise AssertionError(f"{len(failed)} offers failed the fidelity check (strict mode)")

## Deliverable

| File | Content |
|---|---|
| `data/raw/{AG}.txt` | Vision-normalized, structured text (artifact for Stage 2) |
| `data/raw/{AG}.ref.txt` | pdfplumber `layout=False` reference (fidelity check, audit) |
| `data/raw/{AG}.json` | Status: `clean`, `n_pages`, `chars`, source PDF |

**Next:** Stage 2 (`02-sanitize-extract.ipynb`) reads `data/raw/{AG}.txt`,
applies the regex PII sanitizer + vision customer-PII masking and extracts
`datum`/`preis` → `data/redacted/text/` + `data/extracted/`.
